In [1]:
import json
import os
from google.colab import drive
import joblib
from lightgbm import LGBMClassifier
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

In [2]:
from google.colab import drive

drive.mount('/content/drive')

# Updated path to remove duplicate directory bug
data_dir = '/content/drive/MyDrive/AML_Dataset'

print('Loading leakage-free datasets...')
train_df = pd.read_parquet(os.path.join(data_dir, 'train_features.parquet'))
test_df = pd.read_parquet(os.path.join(data_dir, 'test_features.parquet'))

print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')

Mounted at /content/drive
Loading leakage-free datasets...
Train shape: (4062676, 25)
Test shape: (1015669, 25)


In [3]:
# Target and non-predictive metadata columns
target_col = 'Is Laundering'
drop_cols = ['Timestamp', 'Account', 'Account.1', target_col]

# Categorical columns that need encoding
categorical_cols = [
    'From Bank',
    'To Bank',
    'Payment Format',
    'Payment Currency',
    'Receiving Currency',
]

print('Encoding categorical variables...')
category_mappings = {}

for col in categorical_cols:
  if col in train_df.columns:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')

    category_mappings[col] = dict(enumerate(train_df[col].cat.categories))

# Separate features (X) and target (y)
X_train = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])
y_test = test_df[target_col]

# Ensure exact feature column alignment
feature_names = list(X_train.columns)
X_test = X_test[feature_names]

Encoding categorical variables...


In [18]:
print('Training tuned LightGBM baseline model...')

# Unweighted training gives the highest PR-AUC and realistic precision-recall trade-offs
model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=8,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

model.fit(X_train, y_train)
print('Model training complete.')

Training tuned LightGBM baseline model...
Model training complete.


In [19]:
print('Evaluating model performance...')
y_pred_proba = model.predict_proba(X_test)[:, 1]

pr_auc = average_precision_score(y_test, y_pred_proba)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print('\n=== Model Metrics ===')
print(f'PR-AUC  (Primary Metric): {pr_auc:.4f}')
print(f'ROC-AUC (Secondary Metric): {roc_auc:.4f}')

Evaluating model performance...

=== Model Metrics ===
PR-AUC  (Primary Metric): 0.1188
ROC-AUC (Secondary Metric): 0.8207


In [20]:
# Search dynamic decision thresholds across the full standard spectrum
best_thresh = 0.50
best_f1 = 0.0

thresholds = np.linspace(0.01, 0.99, 100)

for thresh in thresholds:
  preds = (y_pred_proba >= thresh).astype(int)
  report = classification_report(y_test, preds, output_dict=True, zero_division=0)
  f1 = report['1']['f1-score']
  if f1 > best_f1:
    best_f1 = f1
    best_thresh = thresh

print(f'\nOptimal Decision Threshold: {best_thresh:.4f}')
print(f'Best F1-Score: {best_f1:.4f}')
print('\nClassification Report (At Optimal Threshold):')
print(classification_report(y_test, (y_pred_proba >= best_thresh).astype(int), zero_division=0))

# Save trained model artifact
joblib.dump(model, os.path.join(data_dir, 'lgbm_aml_model.pkl'))
print(f'\nModel saved successfully to {data_dir}/lgbm_aml_model.pkl')


Optimal Decision Threshold: 0.9900
Best F1-Score: 0.2383

Classification Report (At Optimal Threshold):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1013872
           1       0.18      0.35      0.24      1797

    accuracy                           1.00   1015669
   macro avg       0.59      0.67      0.62   1015669
weighted avg       1.00      1.00      1.00   1015669


Model saved successfully to /content/drive/MyDrive/AML_Dataset/lgbm_aml_model.pkl


In [21]:
# Export metadata for Person 4 downstream inference
metadata = {
    'optimal_threshold': float(best_thresh),
    'primary_pr_auc': float(pr_auc),
    'secondary_roc_auc': float(roc_auc),
    'feature_names': feature_names,
    'category_mappings': category_mappings
}

with open(os.path.join(data_dir, 'model_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=4)

print(f"Model metadata saved successfully to {data_dir}/model_metadata.json")

Model metadata saved successfully to /content/drive/MyDrive/AML_Dataset/model_metadata.json
